# 13 - Advanced DataLoader practice

**Time:** about 35-45 minutes. **Before this:** lessons 06, 07, and 11. Use Python 3.11 or later.

You will tune a small loader, check that it returns the right samples, and make random choices repeatable. You will also separate the time spent waiting for data, moving tensors, and computing.

The examples create tiny tensors locally. There are no downloads. Every required experiment works on a CPU; a CUDA experiment is optional. Timings vary by machine.

An **epoch** is one complete pass through the data. **Throughput** is completed samples per second. A **random seed** chooses the starting point of a sequence of random numbers.

Previous: [12 - Combining concurrency mechanisms](12_combining_concurrency_mechanisms.ipynb). Course guide: [README](README.md).


**MLE** means machine learning engineer. **CUDA** is the software PyTorch uses for NVIDIA GPUs. A **tensor** is an array of numbers. A three-channel RGB image stores red, green, and blue values at each pixel.

## Set up

PyTorch is optional. If it is missing, this notebook skips its experiments and you can still read the explanations and inspect the code. Install it in the notebook's environment using the printed command, restart the kernel, then run all cells.

Worker examples run a helper file in a fresh Python process. Windows uses `spawn`: each child starts Python and imports the helper. The helper has top-level dataset definitions and an `if __name__ == "__main__"` guard, as in lesson 03. [PyTorch's platform notes](https://docs.pytorch.org/docs/stable/data.html#platform-specific-behaviors).


The complete [helper file](lessons/advanced_dataloader_demo.py) contains the runnable experiment. The excerpts below show the parts needed for each concept.

In [1]:
from pathlib import Path
import ast
import importlib.util
import json
import subprocess
import sys
from IPython.display import Code, display

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "lessons" / "advanced_dataloader_demo.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the course folder.")
HELPER = ROOT / "lessons" / "advanced_dataloader_demo.py"
DEPENDENCIES_READY = all(
    importlib.util.find_spec(name) is not None for name in ("torch", "numpy")
)
if sys.version_info < (3, 11):
    raise RuntimeError("Use Python 3.11 or later for this lesson.")
print("Python:", sys.version.split()[0])
print("PyTorch experiments available:", DEPENDENCIES_READY)
if not DEPENDENCIES_READY:
    print(
        ('& ' if sys.platform == 'win32' else '')
        + f'"{sys.executable}" -m pip install '
        f'-r "{ROOT / "requirements-advanced.txt"}" '
        f'-r "{ROOT / "requirements-torch.txt"}"'
    )

def show_section(name):
    source = HELPER.read_text(encoding="utf-8")
    section = source.split(f"# SECTION: {name}\n", 1)[1]
    section = section.split(f"# END SECTION: {name}", 1)[0]
    display(Code(section.strip(), language="python"))

def show_definition(name):
    source = HELPER.read_text(encoding="utf-8")
    node = next(node for node in ast.parse(source).body
                if getattr(node, "name", None) == name)
    display(Code(ast.get_source_segment(source, node), language="python"))


def run_helper(*arguments):
    if not DEPENDENCIES_READY:
        print("Skipped: install the optional dependencies to run this example.")
        return None
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", str(HELPER), *map(str, arguments)],
        cwd=ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        timeout=240,
    )
    if completed.returncode:
        raise RuntimeError(
            f"Helper exited with code {completed.returncode}:\n"
            + completed.stdout + "\n" + completed.stderr
        )
    lines = completed.stdout.splitlines()
    for line in lines:
        if not line.startswith("RESULT_JSON="):
            print(line)
    if completed.stderr:
        print(completed.stderr)
    result_line = next((line for line in lines if line.startswith("RESULT_JSON=")), None)
    if result_line is None:
        raise RuntimeError("Helper returned no result:\n" + completed.stdout + completed.stderr)
    return json.loads(result_line.split("=", 1)[1])


Python: 3.13.7
PyTorch experiments available: True


## 1. Know what each setting changes

| Setting | What it controls | Main tradeoff |
| --- | --- | --- |
| `batch_size` | Samples grouped into one batch | Larger batches use more memory and change the model's training behavior |
| `num_workers` | Processes preparing samples | More preparation capacity, plus startup and process memory |
| `prefetch_factor` | Batches scheduled ahead per worker | More prepared work can reduce stalls, but needs more memory |
| `persistent_workers` | Whether workers stay alive between epochs | Less repeated startup, with worker memory and state kept alive |
| `pin_memory` | Whether returned CPU tensors use pinned memory | Can help CUDA transfers; keeps that CPU memory in RAM |

Pinned memory is CPU memory that the operating system keeps in place for device transfers. It does not move a tensor to the GPU. See the [DataLoader arguments](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) and [transfer guide](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html).

**Predict:** Will two workers necessarily beat zero workers for a dataset that creates tiny tensors?


## 2. Inspect the data before timing it

Each sample contains a generated `3 × 16 × 16` image and its index. The dataset creates samples on demand rather than storing a large list.

The short `sleep` stands in for slow preparation. It makes the overlap easy to observe. It does not measure image decoding or the speed of CPU transforms.

This is a map-style dataset: the loader assigns requested indices to workers. The helper checks that every index appears exactly once in every epoch. For an iterable dataset, worker replicas need their own partitioning, as you saw in lesson 06. [Dataset types](https://docs.pytorch.org/docs/stable/data.html#dataset-types).


In [2]:
show_section("dataset")


class TinySamples(Dataset):
    """Create each sample when requested; keep no large list in worker memory."""

    def __init__(self, count=64, delay=0.004, random_draws=False):
        self.count = count
        self.delay = delay
        self.random_draws = random_draws

    def __len__(self):
        return self.count

    def __getitem__(self, index):
        sleep(self.delay)  # Stand-in for loading/preparation; not real image decoding.
        sample = {
            "image": torch.full((3, 16, 16), index / self.count, dtype=torch.float32),
            "index": index,
        }
        if self.random_draws:
            worker = get_worker_info()
            sample.update(
                worker=-1 if worker is None else worker.id,
                python_draw=random.random(),
                numpy_draw=float(np.random.random()),
                torch_draw=float(torch.rand(())),
            )
        return sample

## 3. Separate startup from later epochs

The helper measures two full epochs for each configuration. Every trial uses the same 64 samples and the same preparation and consumer delay per sample. Changing batch size does not change the total simulated consumer work.

Epoch 1 includes worker startup and imports. Epoch 2 shows the effect of keeping workers alive. Final cleanup of a persistent loader happens after the epoch timers; its shutdown is excluded from those times. Two epochs are an introduction to measurement; they are not enough to prove stable performance. Repeat promising configurations and vary their order before drawing a strong conclusion.

`first_batch_seconds` includes iterator creation and waiting for the first batch. `median_later_wait_ms` measures how long `next(iterator)` waits after the first batch. Some work happens ahead of that call, so a short wait does not mean preparation was free.

The helper and its workers limit library threads to one, following lesson 11. We vary loader workers while keeping that setting fixed. [PyTorch's tuning guide](https://docs.pytorch.org/tutorials/recipes/recipes/tuning_guide.html#enable-asynchronous-data-loading-and-augmentation).


The excerpt focuses on one epoch. The helper builds a loader, calls this function twice, and then releases the loader. A **configuration** is the chosen set of settings, such as worker count and batch size.

In [3]:
show_definition("read_epoch")

def read_epoch(loader, sample_count, consumer_delay):
    """Time a complete pass and check that every index was returned once."""
    start = perf_counter()
    iterator = iter(loader)
    indices, waits = [], []
    first_batch_seconds = None
    try:
        while True:
            wait_start = perf_counter()
            try:
                batch = next(iterator)
            except StopIteration:
                break
            waits.append(perf_counter() - wait_start)
            if first_batch_seconds is None:
                first_batch_seconds = perf_counter() - start
            indices.extend(batch["index"].tolist())
            # Same amount of simulated consumer work per sample in every trial.
            sleep(consumer_delay * len(batch["index"]))
    finally:
        del iterator
    seconds = perf_counter() - start
    assert sorted(indices) == list(range(sample_count)), "Missing or duplicate sample"
    return {
        "seconds": seconds,
        "samples_per_second": sample_count / seconds,
        "first_batch_seconds": first_batch_seconds,
        "median_later_wait_ms": 1000 * median(waits[1:]) if len(waits) > 1 else 0,
        "samples": len(indices),
    }

**Predict:** Which configuration will have the largest difference between its first and second epoch? Does doubling prefetch guarantee a throughput improvement?

**Run:** The cell below launches several small worker groups. Allow roughly a minute on machines with slow Python/PyTorch imports.


In [4]:
benchmark_result = run_helper("--mode", "benchmark")
if benchmark_result is not None and benchmark_result["status"] == "ok":
    trials = benchmark_result["benchmark"]
    print("\nworkers batch prefetch persistent epoch seconds samples/s first batch(s) later wait(ms)")
    for trial in trials:
        config = trial["config"]
        for epoch_number, epoch in enumerate(trial["epochs"], start=1):
            print(
                f'{config["workers"]:7} {config["batch_size"]:5} '
                f'{config["prefetch"]:8} {str(config["persistent"]):10} '
                f'{epoch_number:5} {epoch["seconds"]:7.3f} '
                f'{epoch["samples_per_second"]:9.1f} '
                f'{epoch["first_batch_seconds"]:14.3f} '
                f'{epoch["median_later_wait_ms"]:14.3f}'
            )


Running: {'workers': 0, 'batch_size': 4, 'prefetch': 1, 'persistent': False}
Running: {'workers': 2, 'batch_size': 4, 'prefetch': 1, 'persistent': False}
Running: {'workers': 2, 'batch_size': 4, 'prefetch': 1, 'persistent': True}
Running: {'workers': 2, 'batch_size': 4, 'prefetch': 2, 'persistent': True}
Running: {'workers': 2, 'batch_size': 8, 'prefetch': 2, 'persistent': True}

workers batch prefetch persistent epoch seconds samples/s first batch(s) later wait(ms)
      0     4        1 False          1   0.417     153.7          0.019         17.533
      0     4        1 False          2   0.416     153.8          0.018         17.629
      2     4        1 False          1   1.709      37.4          1.204          0.896
      2     4        1 False          2   1.693      37.8          1.194          0.904
      2     4        1 True           1   1.362      47.0          1.213          0.877
      2     4        1 True           2   0.167     382.1          0.019          0.797
 

**Explain:** Compare adjacent configurations so you change one setting at a time.

- Zero workers may win because the data is tiny and process startup is expensive.
- Persistent workers can improve later epochs by avoiding another round of startup.
- Extra prefetch helps when the consumer would otherwise wait. It cannot make a slow consumer faster.
- Larger batches may reduce per-batch overhead. Here the consumer delay is deliberately proportional to sample count.

These are explanations to check against your own output, rather than promises about which row will win. For an actual MLE workload, use its real decode, transforms, batch consumption, and sample sizes.


## 4. Estimate prefetch memory before increasing it

With automatic batching, a useful planning estimate is:

`workers × prefetch_factor × batch_size × bytes_per_sample`

This estimates the tensor payload if those scheduled batches have been built in memory. It is **not a hard limit on total process memory**. Add room for each worker's Python/PyTorch runtime, dataset state, temporary transforms, the current batch, metadata, and possible pinned copies.

The loader controls how much work is scheduled ahead, as discussed in [the prefetch argument](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader). Backpressure controls a queue; memory planning must also account for what each queued item holds.

**Predict:** What happens to this estimate if you double both batch size and prefetch?


In [5]:
def prefetch_payload_mib(workers, prefetch, batch_size, shape, bytes_per_element=4):
    elements_per_sample = 1
    for dimension in shape:
        elements_per_sample *= dimension
    return workers * prefetch * batch_size * elements_per_sample * bytes_per_element / 2**20

for prefetch in (1, 2):
    print(
        f"4 workers, prefetch={prefetch}, batch=32, float32 3x224x224: "
        f"{prefetch_payload_mib(4, prefetch, 32, (3, 224, 224)):.1f} MiB"
    )

if benchmark_result is not None and benchmark_result["status"] == "ok":
    print("\nTiny experiment payload estimates:")
    for trial in benchmark_result["benchmark"]:
        print(trial["config"], f'{trial["prefetch_tensor_payload_mib"]:.3f} MiB')


4 workers, prefetch=1, batch=32, float32 3x224x224: 73.5 MiB
4 workers, prefetch=2, batch=32, float32 3x224x224: 147.0 MiB

Tiny experiment payload estimates:
{'batch_size': 4, 'persistent': False, 'prefetch': 1, 'workers': 0} 0.000 MiB
{'batch_size': 4, 'persistent': False, 'prefetch': 1, 'workers': 2} 0.023 MiB
{'batch_size': 4, 'persistent': True, 'prefetch': 1, 'workers': 2} 0.023 MiB
{'batch_size': 4, 'persistent': True, 'prefetch': 2, 'workers': 2} 0.047 MiB
{'batch_size': 8, 'persistent': True, 'prefetch': 2, 'workers': 2} 0.094 MiB


**Explain:** Doubling both settings multiplies the estimate by four. Our tiny tensors barely use payload memory, so their runtime overhead can dominate. Real images and large batches change that balance.

A loader with zero workers has no worker prefetch queue. It still holds the current batch and its dataset. An estimate of zero for the queue therefore does not mean zero memory use.


## 5. Make worker randomness repeatable

A loader's `torch.Generator` supplies seeds and, when shuffling, index order. A top-level `worker_init_fn` uses the worker's PyTorch seed to initialize Python's `random` and NumPy as well.

We seed all three because real transforms may use different libraries. The worker function runs after PyTorch has seeded that worker. With zero workers, seed the main process instead. This follows [PyTorch's reproducibility example](https://docs.pytorch.org/docs/stable/notes/randomness.html#dataloader).

Inspect the actual helper code:


**RNG** means random-number generator: an object that produces a sequence of numbers from a seed. `torch.Generator()` creates one for the loader. `worker_init_fn` is a function the loader calls once when each worker starts. The `% (2**32)` expression makes its seed fit the range NumPy accepts.

In [6]:
show_section("seeds")

def seed_worker(worker_id):
    """Initialize each library's sequence of random numbers for this worker."""
    # PyTorch has already set this worker's seed. NumPy accepts a 32-bit seed.
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.set_num_threads(1)


def make_loader(dataset, workers, batch_size, prefetch=1, persistent=False,
                seed=123, shuffle=False, pin_memory=False):
    generator = torch.Generator().manual_seed(seed)
    options = dict(
        batch_size=batch_size,
        num_workers=workers,
        shuffle=shuffle,
        drop_last=False,
        generator=generator,
        pin_memory=pin_memory,
    )
    if workers:
        options.update(
            multiprocessing_context="spawn",
            worker_init_fn=seed_worker,
            prefetch_factor=prefetch,
            persistent_workers=persistent,
            timeout=60,
        )
    else:
        # With no worker processes, worker_init_fn would not run.
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
    return DataLoader(dataset, **options)

The random dataset returns one draw from each library. These stand in for choices such as a crop location or image brightness; the generated image itself stays unchanged.

**Predict:** If we create a fresh loader twice with the same seed and settings, will its two-epoch results repeat? With persistent workers, will epoch 2 reuse epoch 1's random draws?


In [7]:
randomness_result = run_helper("--mode", "randomness")
if randomness_result is not None and randomness_result["status"] == "ok":
    experiment = randomness_result["randomness"]
    print("Two fresh runs match:", experiment["same_two_epoch_runs"])
    print("Random draws change across epochs:", experiment["different_epoch_draws"])
    for epoch, (order, rows) in enumerate(
        zip(experiment["epoch_orders"], experiment["first_three_rows_per_epoch"]), start=1
    ):
        print(f"\nEpoch {epoch} sample order:", order)
        for row in rows:
            print(
                f'index={row["index"]:2} worker={row["worker"]} '
                f'Python={row["python_draw"]:.4f} '
                f'NumPy={row["numpy_draw"]:.4f} torch={row["torch_draw"]:.4f}'
            )


Two fresh runs match: True
Random draws change across epochs: True

Epoch 1 sample order: [6, 5, 2, 8, 10, 3, 11, 0, 7, 4, 9, 1]
index= 6 worker=0 Python=0.7731 NumPy=0.4093 torch=0.7683
index= 5 worker=0 Python=0.6670 NumPy=0.7467 torch=0.8738
index= 2 worker=0 Python=0.9887 NumPy=0.7627 torch=0.1450

Epoch 2 sample order: [7, 8, 3, 2, 4, 5, 6, 0, 11, 10, 9, 1]
index= 7 worker=0 Python=0.2772 NumPy=0.3464 torch=0.1149
index= 8 worker=0 Python=0.2872 NumPy=0.1013 torch=0.4119
index= 3 worker=0 Python=0.4505 NumPy=0.1923 torch=0.0735


**Explain:** Both fresh two-epoch runs match in this example. Within one run, persistent workers keep advancing their random-number sequences.

The worker initializer runs when a worker starts; it does not run again at every epoch when that worker persists. Shuffling in the main process also uses an advancing generator. Resetting a main-process seed does not reset already-running worker state. See [persistent workers](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) and [worker initialization](https://github.com/pytorch/pytorch/blob/main/torch/utils/data/_utils/worker.py).

A fixed seed does not guarantee identical transforms after changing worker count or batch assignment: a sample may receive a different position in a different worker's random sequence. If you need each sample's choices to stay the same across such settings, derive a local random seed from the run seed, epoch, and sample index. Avoid relying on Python's built-in `hash()`, which can vary across processes.

Repeatability also depends on library versions, device, and operations. This experiment checks random draws and sample coverage; it does not prove that all model training is deterministic. [Reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness.html).


Persistent workers also keep their own dataset objects. Changing `dataset.delay` or another field in the main process does not automatically update those worker copies. If workers need new state each epoch, communicate it explicitly or recreate them.

## 6. Time data wait, movement, and compute separately

A fast loader alone does not guarantee a fast training loop. There are three stages to inspect:

`wait for batch → move tensor to device → compute`

The helper warms up workers and the device, then times a tiny matrix multiplication. The CPU version moves a CPU tensor to the CPU, which often returns the same tensor. That timing is not a GPU transfer estimate.

On CUDA, operations usually return to Python before GPU work finishes. `torch.cuda.synchronize()` waits for queued device work so each stage's elapsed time is meaningful. These barriers intentionally remove transfer/compute overlap. Treat the results as a diagnostic breakdown, not an optimized training benchmark. [CUDA synchronization](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html).

**Predict:** What would a large data-wait total suggest? What does a very small transfer total on CPU tell you?


**Warm-up** means running a short pass before timing, so startup happens first. A **barrier** is a point that waits for earlier work to finish; the CUDA synchronization calls are barriers here. **Inference** means computing outputs without training the model. `torch.inference_mode()` skips recording information used to calculate gradients, and `@` performs matrix multiplication.

In [8]:
show_section("profiling")
cpu_profile_result = run_helper("--mode", "profile", "--device", "cpu")
if cpu_profile_result is not None and cpu_profile_result["status"] == "ok":
    profile = cpu_profile_result["profile"]
    print("CPU batches:", profile["batches"])
    for stage, milliseconds in profile["totals_ms"].items():
        print(f"{stage:8}: {milliseconds:.3f} ms")


def stage_profile(device="cpu", pin_memory=False):
    """Use barriers to separate stages. This intentionally removes GPU overlap."""
    if device == "cuda" and not torch.cuda.is_available():
        return {"status": "skipped", "reason": "CUDA is not available"}

    target = torch.device(device)
    # Make weights for a tiny matrix multiplication and warm up the device.
    # inference_mode skips recording information used to calculate gradients.
    weights = torch.linspace(0, 1, 3 * 16 * 16 * 16, device=target).reshape(-1, 16)
    with torch.inference_mode():
        warmup = torch.zeros((4, 3 * 16 * 16), device=target) @ weights
    if device == "cuda":
        torch.cuda.synchronize()
    del warmup

    loader = make_loader(
        TinySamples(count=32, delay=0.004), workers=2, batch_size=4,
        persistent=True, pin_memory=pin_memory,
    )
    rows = []
    try:
        # Finish one warm-up pass so worker startup is outside the stage totals.
        for batch in loader:
            pass
        iterator = iter(loader)
        indices = []
        try:
            with torch.inference_mode():
                while True:
                    start = perf_counter()
                    try:
                        batch = next(iterator)
                    except StopIteration:
                        break
                    wait = perf_counter() - start
                    indices.extend(batch["index"].tolist())

                    start = perf_counter()
                    images = batch["image"].to(target, non_blocking=(device == "cuda"))
                    if device == "cuda":
                        torch.cuda.synchronize()
                    transfer = perf_counter() - start

                    start = perf_counter()
                    output = images.flatten(1) @ weights
                    if device == "cuda":
                        torch.cuda.synchronize()
                    compute = perf_counter() - start
                    rows.append(dict(wait=wait, transfer=transfer, compute=compute))
                    # This use happens outside timing, after CUDA synchronization.
                    assert bool(torch.isfinite(output).all())
            assert sorted(indices) == list(range(32))
        finally:
            del iterator
    finally:
        del loader
        gc.collect()
    return {
        "status": "ok", "device": device, "pin_memory": pin_memory,
        "batches": len(rows),
        "totals_ms": {key: 1000 * sum(row[key] for row in rows) for key in rows[0]},
    }

CPU batches: 8
compute : 0.830 ms
transfer: 2.395 ms
wait    : 70.359 ms


**Explain:** A large wait total suggests that the consumer often runs out of prepared data. Investigate loading and transforms, then worker/prefetch settings. A large compute total suggests the consumer may already be the bottleneck.

The required experiment uses tiny tensors, so overhead and timing noise can outweigh actual compute. Stage times also exclude some Python work and assertions; their sum is not the full program runtime.

For a CUDA machine, the next cell is optional. `pin_memory=True` prepares CPU tensors for transfers, while `.to("cuda", non_blocking=True)` allows Python to continue after scheduling the copy. A **CUDA stream** is an ordered queue of GPU operations. Overlap between transfer and computation needs suitable hardware and stream scheduling; this example waits for each stage to finish before timing the next one. [Pinned memory and nonblocking copies](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html).


In [9]:
RUN_CUDA_EXPERIMENT = False  # Set True only if you want to run it.

if RUN_CUDA_EXPERIMENT and DEPENDENCIES_READY:
    cuda_profile_result = run_helper(
        "--mode", "profile", "--device", "cuda", "--pin-memory"
    )
    print(cuda_profile_result["profile"])
else:
    print("Optional CUDA experiment skipped. The CPU lesson is complete.")


Optional CUDA experiment skipped. The CPU lesson is complete.


## 7. Finish loaders deliberately

The examples consume each epoch fully, then drop the loader in a `finally` block. Persistent workers remain available between epochs; keeping the loader alive keeps their runtime and dataset state alive too.

The helper deletes the loader and asks Python to collect unused objects after it is finished. It uses no private PyTorch shutdown methods. Running each experiment in a subprocess also gives it a clear end of lifetime.

In a long-running notebook, avoid keeping unnecessary references to old loaders or iterators. If you stop halfway through an epoch, worker cleanup and queued work need extra care. The examples here finish their small, finite passes.


## Exercises

Try these before opening the answers.

1. Add a one-worker trial with batch size 4 and prefetch 1. Compare both epochs with zero workers and with the persistent two-worker trial. Repeat it once. Explain whether your measurements support a worker choice.
2. Estimate the prefetched float32 payload for two workers, prefetch 2, batch size 16, and `3 × 224 × 224` images. Name three other memory costs.
3. Explain why a fixed seed can reproduce two runs but still give different random choices after changing the worker count.
4. A synchronized CUDA profile spends most of its time in compute. Predict what doubling prefetch will do. Explain what the profile cannot tell you about overlap.

Use this optional cell for exercise 1. Set the flag to `True` when you are ready.


In [10]:
RUN_EXTRA_TRIAL = False
if RUN_EXTRA_TRIAL:
    one_worker_result = run_helper(
        "--mode", "single", "--workers", 1, "--batch-size", 4,
        "--prefetch", 1, "--persistent",
    )
    if one_worker_result is not None and one_worker_result.get("status") == "ok":
        print(one_worker_result["benchmark"][0]["epochs"])
else:
    print("Exercise trial is ready when you set RUN_EXTRA_TRIAL = True.")


Exercise trial is ready when you set RUN_EXTRA_TRIAL = True.


<details>
<summary>Worked answers</summary>

1. Run the optional cell twice. Compare epoch 1 with epoch 1 and epoch 2 with epoch 2. The provided one-worker trial uses persistence, so compare it with the persistent two-worker trial. Do not assert a timing winner: worker startup may dominate and small timing differences can be noise. A reasonable choice needs repeated measurements on the workload you care about.
2. The estimate is `2 × 2 × 16 × 3 × 224 × 224 × 4 / 2**20 = 36.75 MiB`. Worker runtimes, temporary transform buffers, dataset state, current batches, and pinned copies can add to it.
3. Identical seeds and settings reproduce the assignment of this example's random sequences. Changing worker count can assign a sample to another sequence or another position in that sequence. A local seed based on epoch and sample index can make that sample's choices independent of worker assignment.
4. If batches already arrive promptly, extra prefetch is unlikely to reduce compute time. It may mostly add memory use. The synchronization barriers remove overlap, so the profile does not show the best throughput achievable when transfers and compute overlap.

</details>


## Check your understanding

You should now be able to explain why the first epoch can be slow, tune one loader setting at a time, budget prefetch payload, and explain why persistent workers keep advancing their random sequences after the initial seed.

For an MLE interview, describe a measured experiment: the fixed workload, the setting you changed, the correctness checks, the timing result, and the memory tradeoff.

Official references: [DataLoader](https://docs.pytorch.org/docs/stable/data.html), [reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html), [performance tuning](https://docs.pytorch.org/tutorials/recipes/recipes/tuning_guide.html), and [CPU-to-CUDA transfers](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html).
